**Система:** учет собак и кормов  
**База данных:** SQLite  
**ORM:** SQLAlchemy  
**Таблицы:** `dogs` и `dog_foods`  
**Операции:** поиск по кличке, сортировка по возрасту, фильтрация по названию корма и расчет минимальной цены

Проект создает локальную базу данных, заполняет ее демонстрационными
записями и выполняет основные запросы через ORM.

## 0. Установка и импорт библиотек

In [ ]:
!pip -q install sqlalchemy

In [ ]:
from sqlalchemy import (
    create_engine,
    Column,
    Integer,
    String,
    Float,
    ForeignKey,
    func
)
from sqlalchemy.orm import declarative_base, sessionmaker, relationship

## 1. ORM-модели

In [ ]:
Base = declarative_base()


class DogFood(Base):
    __tablename__ = "dog_foods"

    code = Column(Integer, primary_key=True)
    name = Column(String(100), nullable=False)

    def __repr__(self):
        return f"DogFood(code={self.code}, name='{self.name}')"


class Dog(Base):
    __tablename__ = "dogs"

    id = Column(Integer, primary_key=True, autoincrement=True)
    name = Column(String(50), nullable=False)
    gender = Column(String(10), nullable=False)
    age = Column(Integer, nullable=False)
    food_code = Column(
        Integer,
        ForeignKey("dog_foods.code"),
        nullable=False
    )
    food_price = Column(Float, nullable=False)

    food = relationship("DogFood", backref="dogs")

    def __repr__(self):
        return (
            f"Dog(id={self.id}, name='{self.name}', "
            f"gender='{self.gender}', age={self.age}, "
            f"food_code={self.food_code}, "
            f"food_price={self.food_price})"
        )

## 2. Создание базы данных

In [ ]:
engine = create_engine(
    "sqlite:///dogs_database.db",
    echo=False
)

Base.metadata.create_all(engine)

Session = sessionmaker(bind=engine)
session = Session()

print("База данных создана.")

## 3. Начальные данные

In [ ]:
foods = [
    DogFood(code=1, name="Royal Canin для щенков"),
    DogFood(code=2, name="1st Choice"),
    DogFood(code=3, name="Acana Adult"),
    DogFood(code=4, name="Brit Care"),
    DogFood(code=5, name="Hills Science Diet"),
    DogFood(code=6, name="Monge"),
    DogFood(code=7, name="Clan Classic"),
    DogFood(code=8, name="Britt Care для щенков"),
    DogFood(code=9, name="Pronature"),
    DogFood(code=10, name="Pro Plan"),
]

dogs = [
    Dog(
        name="Барсик",
        gender="м",
        age=3,
        food_code=1,
        food_price=1500.0
    ),
    Dog(
        name="Шарик",
        gender="м",
        age=5,
        food_code=2,
        food_price=2000.0
    ),
    Dog(
        name="Рокки",
        gender="ж",
        age=2,
        food_code=3,
        food_price=1800.0
    ),
    Dog(
        name="Рекс",
        gender="м",
        age=7,
        food_code=9,
        food_price=2200.0
    ),
    Dog(
        name="Персик",
        gender="ж",
        age=2,
        food_code=5,
        food_price=1900.0
    ),
    Dog(
        name="Джаз",
        gender="м",
        age=4,
        food_code=7,
        food_price=2900.0
    ),
    Dog(
        name="Граф",
        gender="м",
        age=9,
        food_code=4,
        food_price=3420.0
    ),
    Dog(
        name="Джони",
        gender="м",
        age=1,
        food_code=6,
        food_price=1270.0
    ),
    Dog(
        name="Линда",
        gender="ж",
        age=7,
        food_code=8,
        food_price=1989.0
    ),
    Dog(
        name="Инфанта",
        gender="ж",
        age=8,
        food_code=10,
        food_price=3980.0
    ),
]

In [ ]:
session.query(Dog).delete()
session.query(DogFood).delete()
session.commit()

session.add_all(foods)
session.commit()

session.add_all(dogs)
session.commit()

print("Собак:", session.query(Dog).count())
print("Кормов:", session.query(DogFood).count())

## 4. Просмотр данных

In [ ]:
for dog in session.query(Dog).all():
    print(
        f"ID {dog.id}: "
        f"{dog.name}, "
        f"пол: {dog.gender}, "
        f"возраст: {dog.age}, "
        f"корм: {dog.food.name}, "
        f"цена: {dog.food_price:.2f}"
    )

## 5. Поиск по кличке

In [ ]:
def find_dog_by_name(name):
    return (
        session.query(Dog)
        .filter(Dog.name.ilike(f"%{name}%"))
        .all()
    )


found = find_dog_by_name("Рекс")

for dog in found:
    print(
        dog.name,
        dog.gender,
        dog.age,
        dog.food.name,
        dog.food_price
    )

## 6. Сортировка по возрасту

In [ ]:
sorted_dogs = (
    session.query(Dog)
    .order_by(Dog.age.desc())
    .all()
)

for dog in sorted_dogs:
    print(
        f"{dog.name}: "
        f"возраст {dog.age}, "
        f"корм {dog.food.name}"
    )

## 7. Фильтрация по названию корма

In [ ]:
def filter_by_food_name(substring):
    return (
        session.query(Dog)
        .join(DogFood)
        .filter(
            ~DogFood.name.ilike(f"%{substring}%")
        )
        .all()
    )


filtered = filter_by_food_name("Care")

for dog in filtered:
    print(
        f"{dog.name}: "
        f"{dog.food.name}, "
        f"{dog.food_price:.2f}"
    )

## 8. Минимальная цена для возрастов, встречающихся не более двух раз

In [ ]:
age_counts = (
    session.query(
        Dog.age,
        func.count(Dog.id).label("count")
    )
    .group_by(Dog.age)
    .having(func.count(Dog.id) <= 2)
    .all()
)

rare_ages = [
    age
    for age, count in age_counts
]

min_price = (
    session.query(func.min(Dog.food_price))
    .filter(Dog.age.in_(rare_ages))
    .scalar()
)

print("Возрастные значения:", rare_ages)
print("Минимальная цена корма:", min_price)

## 9. Добавление новой записи

In [ ]:
def add_dog(
    dog_name,
    gender,
    age,
    food_code,
    food_name,
    food_price
):
    food = (
        session.query(DogFood)
        .filter_by(code=food_code)
        .first()
    )

    if food is None:
        food = DogFood(
            code=food_code,
            name=food_name
        )
        session.add(food)
        session.commit()

    dog = Dog(
        name=dog_name,
        gender=gender,
        age=int(age),
        food_code=food_code,
        food_price=float(food_price)
    )

    session.add(dog)
    session.commit()

    return dog


new_dog = add_dog(
    "Бим",
    "м",
    6,
    11,
    "Новый корм",
    2100
)

print(new_dog)

## 10. Итоговая проверка

In [ ]:
print("Количество собак:", session.query(Dog).count())
print("Количество кормов:", session.query(DogFood).count())

print()
print("Собаки по возрасту:")
for dog in session.query(Dog).order_by(Dog.age.desc()).all():
    print(
        f"{dog.name}: "
        f"{dog.age} лет, "
        f"{dog.food.name}, "
        f"{dog.food_price:.2f}"
    )

## Итог

In [ ]:
print("Реализованные операции:")
print("1. Поиск собаки по кличке")
print("2. Сортировка по возрасту по убыванию")
print("3. Отбор по названию корма")
print("4. Расчет минимальной цены корма")
print("5. Добавление собаки и корма")